In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
import h3
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def calc_diversity(gdf: gpd.GeoDataFrame):
    total_area = gdf['area'].sum()
    gdf_group = gdf.groupby('DESTINO')['area'].sum()
    gdf_group = gdf_group.reset_index()
    gdf_group['area_percentage'] = gdf_group['area'] / total_area
    gdf_group['entropy'] = gdf_group['area_percentage'] * np.log2(gdf_group['area_percentage'])
    shannon_entropy = -gdf_group['entropy'].sum()
    return shannon_entropy

In [9]:
# formato lat, lon
# -36.70690973726542, -73.11671311007053
# -36.71672579573574, -73.10692144785894

bbox = {
    'lat_min': -36.71672579573574,
    'lat_max': -36.70690973726542,
    'lon_min': -73.11671311007053,
    'lon_max': -73.10692144785894,
}

# create a polygon from the bounding box
bbox_polygon = Polygon([
    (bbox['lon_min'], bbox['lat_min']),
    (bbox['lon_min'], bbox['lat_max']),
    (bbox['lon_max'], bbox['lat_max']),
    (bbox['lon_max'], bbox['lat_min'])
])

bbox_polygon = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[bbox_polygon])
bbox_id = 0


In [10]:
scenarios = ['Actual', 'Futuro']

def load_data(scenario):
    landuses = gpd.read_file(f'./data/usos_de_suelo/{scenario}')
    try:
        landuses = landuses.to_crs(epsg=4326)
    except Exception as e:
        print(e)
        pass
    return landuses

output = {}
landuses_hist = {}

for scenario in scenarios:
    landuses = load_data(scenario)
    
    lu_filtered = gpd.overlay(landuses, bbox_polygon, how='intersection')
    lu_filtered.to_crs('EPSG: 32718', inplace=True)
    lu_filtered['area'] = lu_filtered.area
    lu_filtered.to_crs('EPSG: 4326', inplace=True)
    lu_filtered.reset_index(inplace=True)
    lu_filtered['scenario'] = scenario
    
    try:
        landuses_hist.append(lu_filtered)
    except Exception as e:
        landuses_hist[scenario] = lu_filtered
        pass
    diversity = calc_diversity(lu_filtered)
    df_output = pd.DataFrame({
        'bbox_id': bbox_id,
        'diversity': diversity
    }, index=[0])
    output[scenario] = df_output 

In [12]:
import os
indicator_name = 'land_uses_diversity_by_bbox'
value_col = 'diversity'
index_col = 'bbox_id'
format = 'parquet'
scenarios_available = []

save_path = f'./export/'
export_folder = os.path.join(save_path, indicator_name)
os.makedirs(export_folder, exist_ok=True)

for scenario in scenarios:
    scenario_folder = os.path.join(export_folder, scenario)
    os.makedirs(scenario_folder, exist_ok=True)
    try:
        if format == 'parquet':
            export_filename = os.path.join(scenario_folder, f'{indicator_name}.parquet')
            output[scenario].to_parquet(export_filename, index=False)
        elif format == 'shp':
            if os.path.exists(scenario_folder):
                import shutil
                shutil.rmtree(scenario_folder)
            export_filename = os.path.join(scenario_folder, f'{indicator_name}')
            output[scenario].to_file(export_filename)
        scenarios_available.append(scenario)
    except Exception as e:
        print(e)
        pass

indicator_details = {
    'indicator_name': indicator_name,
    'value_col': value_col,
    'index_col': index_col,
    'format': format,
    'scenarios_available': scenarios_available
}
# Save indicator details as .json in the export folder with name same as export_name
import json
details_filename = os.path.join(export_folder, f'indicator_details.json')
with open(details_filename, 'w') as f:
    json.dump(indicator_details, f)